In [1]:
import pandas as pd

df = pd.read_csv("StudentsPerformance.csv")
df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [2]:
data_card = {
    "name": "Students Performance",
    "version": "1.0",
    "domain": "education",
    "created": "2026-01-08",
    "intended_use": "Analyse et recommandation des performances étudiantes",
    "data_source": "Collecté par enquêtes scolaires anonymisées",
    "coverage": "Étudiants — performances académiques",
    "collection_method": "Fichier fourni par un établissement éducatif",
    "period_covered": "Non temporel (instantané)",
}
import yaml
print(yaml.dump(data_card, sort_keys=False))
import json
print(json.dumps(data_card, indent=2))


name: Students Performance
version: '1.0'
domain: education
created: '2026-01-08'
intended_use: "Analyse et recommandation des performances \xE9tudiantes"
data_source: "Collect\xE9 par enqu\xEAtes scolaires anonymis\xE9es"
coverage: "\xC9tudiants \u2014 performances acad\xE9miques"
collection_method: "Fichier fourni par un \xE9tablissement \xE9ducatif"
period_covered: "Non temporel (instantan\xE9)"

{
  "name": "Students Performance",
  "version": "1.0",
  "domain": "education",
  "created": "2026-01-08",
  "intended_use": "Analyse et recommandation des performances \u00e9tudiantes",
  "data_source": "Collect\u00e9 par enqu\u00eates scolaires anonymis\u00e9es",
  "coverage": "\u00c9tudiants \u2014 performances acad\u00e9miques",
  "collection_method": "Fichier fourni par un \u00e9tablissement \u00e9ducatif",
  "period_covered": "Non temporel (instantan\u00e9)"
}


In [3]:
dict_df = pd.DataFrame([
    {
        "column": col,
        "type": str(df[col].dtype),
        "unique_values": df[col].nunique(),
        "sample_values": df[col].dropna().unique()[:5].tolist(),
        "missing_values": df[col].isna().sum()
    }
    for col in df.columns
])
dict_df

,column,type,unique_values,sample_values,missing_values
0,gender,object,2,"[female, male]",0
1,race/ethnicity,object,5,"[group B, group C, group A, group D, group E]",0
2,parental level of education,object,6,"[bachelor's degree, some college, master's deg...",0
3,lunch,object,2,"[standard, free/reduced]",0
4,test preparation course,object,2,"[none, completed]",0
5,math score,int64,81,"[72, 69, 90, 47, 76]",0
6,reading score,int64,72,"[72, 90, 95, 57, 78]",0
7,writing score,int64,77,"[74, 88, 93, 44, 75]",0


In [4]:
# Valeurs manquantes
missing = df.isna().sum()

# Doublons
duplicates = df.duplicated().sum()

# Biais simples (ex: déséquilibre d'une variable catégorique)
gender_bias = df['gender'].value_counts(normalize=True)

missing, duplicates, gender_bias

(gender                         0
 race/ethnicity                 0
 parental level of education    0
 lunch                          0
 test preparation course        0
 math score                     0
 reading score                  0
 writing score                  0
 dtype: int64,
 np.int64(0),
 gender
 female    0.518
 male      0.482
 Name: proportion, dtype: float64)

In [5]:
numeric_stats = df.describe().T[['mean','min','max','50%']]
numeric_stats.rename(columns={'50%': 'median'}, inplace=True)
numeric_stats

,mean,min,max,median
math score,66.089,0.0,100.0,66.0
reading score,69.169,17.0,100.0,70.0
writing score,68.054,10.0,100.0,69.0


In [6]:
df_numeric = df.select_dtypes(include=['number'])
correlation_matrix = df_numeric.corr()
correlation_matrix

,math score,reading score,writing score
math score,1.000000,0.817580,0.802642
reading score,0.817580,1.000000,0.954598
writing score,0.802642,0.954598,1.000000


In [10]:
import pandas as pd
import numpy as np
import json

df = pd.read_csv("StudentsPerformance.csv")

def feature_card_for_column(df: pd.DataFrame, col: str, description: str = "") -> dict:
    s = df[col]
    card = {
        "feature": col,
        "description": description or "",
        "dtype": str(s.dtype),
        "missing_values": int(s.isna().sum()),
        "missing_rate": float(s.isna().mean()),
        "unique_values": int(s.nunique(dropna=True)),
    }

    # Numérique
    if pd.api.types.is_numeric_dtype(s):
        q1 = float(s.quantile(0.25))
        q3 = float(s.quantile(0.75))
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outliers = int(((s < lower) | (s > upper)).sum())

        card.update({
            "unit": "score (0–100)" if "score" in col.lower() else "",
            "min": float(s.min()),
            "max": float(s.max()),
            "mean": float(s.mean()),
            "median": float(s.median()),
            "std": float(s.std()),
            "quartiles": {"q1": q1, "q3": q3, "iqr": float(iqr)},
            "outliers_iqr_count": outliers,
            "sample_values": s.dropna().head(5).tolist()
        })

    # Catégorielle / texte
    else:
        vc = s.value_counts(dropna=True)
        top = vc.head(5)
        card.update({
            "categories_top5": {str(k): int(v) for k, v in top.items()},
            "categories_top5_rate": {str(k): float(v/len(s)) for k, v in top.items()},
            "sample_values": s.dropna().unique()[:5].tolist()
        })

    return card

In [11]:
descriptions = {
    "gender": "Sexe déclaré de l’étudiant",
    "race/ethnicity": "Groupe ethnique déclaré (catégories A–E)",
    "parental level of education": "Niveau d’étude du/des parent(s)",
    "lunch": "Type de repas (standard vs free/reduced)",
    "test preparation course": "Préparation au test (aucune vs complétée)",
    "math score": "Score en mathématiques",
    "reading score": "Score en compréhension écrite",
    "writing score": "Score en expression écrite"
}


In [12]:
feature_cards = {
    col: feature_card_for_column(df, col, descriptions.get(col, ""))
    for col in df.columns
}

print(json.dumps(feature_cards, indent=2, ensure_ascii=False))


{
  "gender": {
    "feature": "gender",
    "description": "Sexe déclaré de l’étudiant",
    "dtype": "object",
    "missing_values": 0,
    "missing_rate": 0.0,
    "unique_values": 2,
    "categories_top5": {
      "female": 518,
      "male": 482
    },
    "categories_top5_rate": {
      "female": 0.518,
      "male": 0.482
    },
    "sample_values": [
      "female",
      "male"
    ]
  },
  "race/ethnicity": {
    "feature": "race/ethnicity",
    "description": "Groupe ethnique déclaré (catégories A–E)",
    "dtype": "object",
    "missing_values": 0,
    "missing_rate": 0.0,
    "unique_values": 5,
    "categories_top5": {
      "group C": 319,
      "group D": 262,
      "group B": 190,
      "group E": 140,
      "group A": 89
    },
    "categories_top5_rate": {
      "group C": 0.319,
      "group D": 0.262,
      "group B": 0.19,
      "group E": 0.14,
      "group A": 0.089
    },
    "sample_values": [
      "group B",
      "group C",
      "group A",
      "group D",

In [13]:
import os
os.makedirs("feature_cards", exist_ok=True)

for col, card in feature_cards.items():
    filename = col.replace("/", "_").replace(" ", "_").lower() + ".json"
    with open(os.path.join("feature_cards", filename), "w", encoding="utf-8") as f:
        json.dump(card, f, indent=2, ensure_ascii=False)